# Double DQN: split the `max` to stop overestimating values

**Paper.** van Hasselt, Guez & Silver, *Deep Reinforcement Learning with Double Q-learning*, AAAI 2016 ([arXiv:1509.06461](https://arxiv.org/abs/1509.06461)), built on top of DQN (Mnih et al., Nature 2015).

**What we build and train.** A minimal DQN on CartPole-v1 (`gymnasium`) that still follows the full protocol, and on top of it the one line the paper contributes: the Double DQN TD target. We train both arms of the ablation end-to-end (vanilla vs Double), measure return and, most importantly, *value overestimation* (predicted minus actual), the quantity from Fig.3 of the paper. Plus a toy demonstration of the mechanism from Fig.1.

**Compute.** CPU, a few minutes for the whole run (3 seeds × 30k steps per arm). No GPU needed: the hyperparameters are scaled down for CPU, the method's protocol is not.

**Parts.**
1. Experiments first: what the paper showed, before the method.
2. Why bother: the overestimation problem and the decoupling trick.
3. Feynman-style theory: where the positive bias of `max` comes from.
4. Setup: model, replay, environment.
5. Part A: the TD target (the core, the paper's contribution).
6. Part B: the `predicted − actual` overestimation diagnostic (the paper's headline).
7. Part C: the toy mechanism of Fig.1, single vs double estimator.
8. Assembling the ablation, the key figure, a controlled contrast to get a feel for the trick.
9. Reflection.

The core of the method is yours to write. The boilerplate (environment, network, replay, training loop, plots) is provided.

## Part 0. Experiments first (read the Results before the method)

Before touching the code, fill in three slots by hand. Open the abstract and Fig.3/Fig.6 of the paper and answer in your own words (this is not graded; it calibrates your expectations).

1. **What they showed versus the abstract.** The claim: DQN systematically *overestimates* Q, and on Atari this hurts. What exactly on their plots supports the claim, the value-estimate curve or the score curve? How did they measure "overestimation"?

2. **The weakest experiment.** Where is the effect least convincing: on which games does Double DQN barely differ from DQN? What does that tell you about when the trick matters and when it does not?

3. **Where it breaks / what is left out.** Double DQN changes exactly one thing in the target and nothing else. What does it NOT fix (exploration, the network, replay)? Where can overestimation still leak in?

Write your answers right here, in this cell.

## Part 1. Why bother

Q-learning bootstraps: the target for $Q(s,a)$ relies on an estimate of the next state's value. In DQN that estimate is $\max_{a'} Q_{\text{target}}(s', a')$. The problem is that the same `max` both **selects** the action and **evaluates** it. And `max` over noisy estimates is a biased operation: $\mathbb{E}[\max_a \hat Q(a)] \ge \max_a \mathbb{E}[\hat Q(a)]$. Noise that happened to push some $\hat Q(a')$ up gets selected and immediately plugged into the target. The bias is systematically upward. Q inflates, and the inflation propagates back through bootstrapping.

**The Double DQN trick is to split the roles.** Let one network **select** the action and another, independent one **evaluate** it. Noise that raised the choice in the first network is no longer correlated with that same choice in the second, and the systematic upward bias goes away. DQN already has a second network for free: the target network. So the whole change is which network you ask for the argmax. One line.

**Why CartPole.** A canonical control benchmark with real environment dynamics, built into gymnasium, runs on CPU in minutes. The return is bounded above (500), which makes overestimation (predicted value above what is achievable) directly observable without Atari scale.

❓ **Inline.** In vanilla DQN the target network was introduced for *stability* (so the target does not chase itself). Double DQN reuses it for a different purpose. Why is having a lagging second network a good fit specifically for decoupling select/evaluate, and what would happen if we asked the target network itself for the argmax?

## Part 2. Theory (Feynman-style): where the bias comes from and where my holes are

**Problem.** We are given a state $s'$ and the true action values $Q^*(s',a')$. We need an estimate of $\max_{a'} Q^*(s',a')$, the value of the best next move, to build the TD target $y = r + \gamma \cdot (\text{this estimate}) \cdot (1-\text{done})$.

**The bias mechanism.** We do not know $Q^*$, only a noisy estimate $\hat Q$. For intuition take the extreme case: all true $Q^*(a')=0$, and the estimates are independent, $\hat Q(a') \sim \mathcal N(0,1)$. Then
$$\mathbb{E}\big[\max_{a'} \hat Q(a')\big] > 0 = \max_{a'} \mathbb{E}[\hat Q(a')],$$
and the gap grows with the number of actions (the maximum of more Gaussians is larger on average). This is overestimation: `max` over noise is biased upward.

**Decoupling.** Take two independent sets of estimates, $\hat Q_A$ and $\hat Q_B$. Select the action with $A$: $a^\star = \arg\max_{a'} \hat Q_A(a')$. Evaluate it with $B$: $\hat Q_B(a^\star)$. Since $B$ took no part in the selection, $\mathbb{E}[\hat Q_B(a^\star)] = 0$ and the bias is gone. This is the double estimator.

In the deep version $A$ = the online network (fresh argmax), $B$ = the target network (evaluation):
$$y^{\text{DQN}} = r + \gamma\, \max_{a'} Q_{\text{target}}(s',a'), \qquad y^{\text{Double}} = r + \gamma\, Q_{\text{target}}\big(s',\, \arg\max_{a'} Q_{\text{online}}(s',a')\big).$$

**What the method measures in practice.** *Value overestimation*: how far what the agent **predicts** for itself ($\max_a Q(s_0,a)$) exceeds what it **actually gets** (the Monte-Carlo return of a greedy rollout from $s_0$). A positive difference is overestimation.

**Where my holes are.** Three core functions: (A) the TD target itself with a `double` flag; (B) the `predicted − actual` diagnostic; (C) the toy mechanism of Fig.1. These are what we write.

❓ **Inline.** In the formula for $y^{\text{Double}}$, why is the argmax taken over `online` and not over `target`? What breaks in the decoupling if both selection and evaluation are given to the target network?

In [ ]:
# setup: deps, environment, model, replay, training loop, eval. Provided, do not modify.
# (if gymnasium is missing:  pip install gymnasium numpy torch)
import copy, math, random
from collections import deque, namedtuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym

DEVICE = torch.device("cpu")
ENV_ID = "CartPole-v1"
Transition = namedtuple("Transition", ("s", "a", "r", "s2", "done"))


class QNet(nn.Module):
    """MLP Q-network: state -> Q-value per action."""
    def __init__(self, obs_dim, n_act, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_act),
        )
    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    """Uniform experience replay: break temporal correlation by random sampling."""
    def __init__(self, cap):
        self.buf = deque(maxlen=cap)
    def push(self, *args):
        self.buf.append(Transition(*args))
    def sample(self, n):
        batch = random.sample(self.buf, n)
        b = Transition(*zip(*batch))
        s = torch.as_tensor(np.array(b.s), dtype=torch.float32, device=DEVICE)
        a = torch.as_tensor(b.a, dtype=torch.long, device=DEVICE).unsqueeze(1)
        r = torch.as_tensor(b.r, dtype=torch.float32, device=DEVICE).unsqueeze(1)
        s2 = torch.as_tensor(np.array(b.s2), dtype=torch.float32, device=DEVICE)
        done = torch.as_tensor(b.done, dtype=torch.float32, device=DEVICE).unsqueeze(1)
        return s, a, r, s2, done
    def __len__(self):
        return len(self.buf)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

print("setup ok:", ENV_ID, DEVICE)

## Part A. The TD target: the core, the paper's contribution

This is the only place where Double DQN differs from vanilla. The `double` flag is the only switch between the arms of the ablation; everything else in training is shared. Implement both branches.

The signature and contract are below. The body is yours. The full recipe, if you get stuck, is in SPOILERS at the bottom.

In [ ]:
def td_target(reward, next_state, done, online, target, gamma, double):
    """Bootstrap TD target y for a minibatch — the quantity Q(s,a) is regressed toward.

    Two arms selected by `double`:
      double=False -> vanilla DQN (Mnih 2015): one network both picks and scores the
                      next action.
      double=True  -> Double DQN (van Hasselt 2016): the choice of the next action and
                      the scoring of it are decoupled across the two given networks.
    A terminal transition contributes no bootstrap term.

    Args:
        reward:     [B,1] float — immediate reward r.
        next_state: [B,obs_dim] float — s'.
        done:       [B,1] float in {0,1} — 1.0 iff s' is terminal.
        online:     QNet — the online (current) network.
        target:     QNet — the lagging target network.
        gamma:      float — discount.
        double:     bool — arm selector (see above).
    Returns:
        [B,1] float — the TD target y. No gradient should flow through it.
    Note: must not require grad through the target (it is a regression label).
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# shapes + no-grad + terminal masking + the defining property of the two arms
probe = QNet(4, 3)
B = 16
rew = torch.zeros(B, 1)
ns = torch.randn(B, 4)
notdone = torch.zeros(B, 1)

y = td_target(rew, ns, notdone, probe, probe, 0.99, double=False)
assert y.shape == (B, 1), f"TD target must be [B,1], got {tuple(y.shape)}"
assert not y.requires_grad, "TD target is a label: no grad should flow through it"

# terminal -> pure reward, no bootstrap
yt = td_target(torch.ones(B, 1), ns, torch.ones(B, 1), probe, probe, 0.99, double=False)
assert torch.allclose(yt, torch.ones(B, 1)), "on terminal (done=1) target must equal reward only"

# When online==target, double picks argmax of the SAME net it evaluates -> identical to max.
# So with a shared net, double and vanilla MUST coincide exactly.
y_van = td_target(rew, ns, notdone, probe, probe, 0.99, double=False)
y_dbl = td_target(rew, ns, notdone, probe, probe, 0.99, double=True)
assert torch.allclose(y_van, y_dbl), \
    "with online==target the double target reduces to the max target (same net selects & evaluates)"

# With DIFFERENT nets the double target gathers target-Q at online's argmax, which is
# <= target's own max -> double target is never above vanilla target (per element).
on, tg = QNet(4, 3), QNet(4, 3)
ya = td_target(rew, ns, notdone, on, tg, 0.99, double=False)
yb = td_target(rew, ns, notdone, on, tg, 0.99, double=True)
assert (yb <= ya + 1e-5).all(), "double target gathers at online-argmax <= target-max, so y_double <= y_vanilla"
print("ok")

In [ ]:
# training driver + epsilon-greedy collection + hard target sync. Provided.
# Takes a snapshot of (online, target) at the moment the target lags furthest behind
# online: the realistic condition under which the max rule overestimates.
def train(double, seed=0, total_steps=30000):
    set_seed(seed)
    env = gym.make(ENV_ID)
    env.reset(seed=seed); env.action_space.seed(seed)
    obs_dim = env.observation_space.shape[0]; n_act = env.action_space.n

    online = QNet(obs_dim, n_act).to(DEVICE)
    target = QNet(obs_dim, n_act).to(DEVICE)
    target.load_state_dict(online.state_dict())
    opt = torch.optim.Adam(online.parameters(), lr=1e-3)

    buf = ReplayBuffer(20000)
    gamma, batch_size, learn_start, target_update = 0.99, 64, 1000, 500
    eps_start, eps_end, eps_decay = 1.0, 0.02, 4000

    snapshot = None; step = 0
    s, _ = env.reset(seed=seed)
    while step < total_steps:
        eps = eps_end + (eps_start - eps_end) * math.exp(-step / eps_decay)
        if random.random() < eps:
            a = env.action_space.sample()
        else:
            with torch.no_grad():
                a = int(online(torch.as_tensor(s, dtype=torch.float32)).argmax().item())
        s2, r, term, trunc, _ = env.step(a)
        done = term  # time-limit truncation is NOT terminal for bootstrapping
        buf.push(s, a, r, s2, float(done))
        s = s2 if not (term or trunc) else env.reset()[0]
        step += 1

        if len(buf) >= learn_start:
            bs, ba, br, bs2, bdone = buf.sample(batch_size)
            y = td_target(br, bs2, bdone, online, target, gamma, double)
            q = online(bs).gather(1, ba)
            loss = F.smooth_l1_loss(q, y)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(online.parameters(), 10.0); opt.step()
            if step % target_update == target_update - 1:
                snapshot = (copy.deepcopy(online), copy.deepcopy(target))
            if step % target_update == 0:
                target.load_state_dict(online.state_dict())
    env.close()
    if snapshot is None:
        snapshot = (copy.deepcopy(online), copy.deepcopy(target))
    return online, target, snapshot


@torch.no_grad()
def eval_return(policy_fn, episodes=20, seed=1000):
    env = gym.make(ENV_ID); returns = []
    for i in range(episodes):
        s, _ = env.reset(seed=seed + i); done = False; ep = 0.0
        while not done:
            s, r, term, trunc, _ = env.step(policy_fn(s)); ep += r; done = term or trunc
        returns.append(ep)
    env.close()
    return float(np.mean(returns)), float(np.std(returns))


def greedy_policy(net):
    @torch.no_grad()
    def pi(s):
        return int(net(torch.as_tensor(s, dtype=torch.float32)).argmax().item())
    return pi


def random_policy(n_act=2):
    def pi(s):
        return random.randrange(n_act)
    return pi

print("train/eval helpers ready")

## Part B. Value overestimation: the paper's headline quantity (Fig.3)

Note: the paper's overestimation is the gap between what the agent **predicts** for itself at the start state and what it **actually gets** by playing greedily from that state. The by-construction "max versus gather" gap inside a single pair of networks is a different quantity (Part B′). A positive value means the agent thinks the state is worth more than it turns out to be.

You have the greedy policy (`greedy_policy`) and the real environment. Implement collecting `predicted` and `actual` and their difference.

In [ ]:
@torch.no_grad()
def value_overestimation(net, gamma=0.99, episodes=20, seed=3000):
    """Paper's headline diagnostic (van Hasselt 2016, Fig.3): how much the agent's OWN
    value estimate at the start state exceeds the value it ACTUALLY realises by acting
    greedily from that same start state.

    For each of `episodes` start states s0 (env reset with seed+i): form the agent's
    predicted value of s0, then play the greedy policy of `net` to termination and form
    the realised (discounted) return from s0. A positive mean gap is overestimation.

    Args:
        net:      QNet — evaluated greedily.
        gamma:    float — discount for the realised return.
        episodes: int — number of start states / rollouts.
        seed:     int — base env seed (episode i uses seed+i).
    Returns:
        (predicted_mean, actual_mean, gap) floats, gap = predicted_mean - actual_mean.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# A trained net should realise a sizeable return; gap = predicted - actual by definition.
# Use a quick-trained net (short budget just to exercise the function).
on_quick, _, _ = train(double=False, seed=0, total_steps=8000)
pm, am, gap = value_overestimation(on_quick, episodes=10)
assert isinstance(pm, float) and isinstance(am, float) and isinstance(gap, float), "must return three floats"
assert abs((pm - am) - gap) < 1e-6, "gap must equal predicted_mean - actual_mean"
assert am > 0.0, "a trained CartPole policy must realise positive discounted return"
# discounted return is bounded by 1/(1-gamma); predicted lives on the same value scale
assert am < 1.0 / (1 - 0.99) + 1.0, "discounted return cannot exceed 1/(1-gamma)"
print(f"predicted={pm:.2f} actual={am:.2f} gap={gap:+.2f}")
print("ok")

## Part B′. The by-construction max-vs-double gap on a shared pair of networks

This is an auxiliary diagnostic: on one fixed `(online, target)` pair and one set of states, how much higher the value under the max rule is than the value under the double rule. It is $\ge 0$ by construction. It shows the bias inside the target itself, separated from training. Implement both value estimates and their difference.

In [ ]:
@torch.no_grad()
def overestimation_gap(online, target, states):
    """On a fixed (online, target) pair and a fixed batch of states, the systematic
    upward bias the max rule injects into the bootstrap value relative to the decoupled
    (Double DQN) rule.

    Form two scalar value estimates averaged over `states`: the value the vanilla-DQN
    target would use, and the value the Double-DQN target would use on the SAME pair.
    Their difference is the bias and is >= 0 by construction.

    Args:
        online: QNet — selects the action in the decoupled estimate.
        target: QNet — scores the action in both estimates.
        states: [N,obs_dim] float — states to average over.
    Returns:
        (dqn_val, double_val, gap) floats, gap = dqn_val - double_val (>= 0).
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
@torch.no_grad()
def collect_states(net, episodes=30, seed=2000):
    env = gym.make(ENV_ID); pi = greedy_policy(net); states = []
    for i in range(episodes):
        s, _ = env.reset(seed=seed + i); done = False
        while not done:
            states.append(s); s, r, term, trunc, _ = env.step(pi(s)); done = term or trunc
    env.close()
    return torch.as_tensor(np.array(states), dtype=torch.float32)

on, tg = QNet(4, 3), QNet(4, 3)
st = torch.randn(200, 4)
dv, ddv, gap = overestimation_gap(on, tg, st)
assert abs((dv - ddv) - gap) < 1e-6, "gap must equal dqn_val - double_val"
assert gap >= -1e-6, "max-rule value >= decoupled value by construction (gap >= 0)"
# with online==target the two estimates coincide -> gap == 0
dv2, ddv2, gap2 = overestimation_gap(on, on, st)
assert abs(gap2) < 1e-6, "with online==target the gap must be exactly 0 (same net selects & evaluates)"
print(f"dqn_val={dv:.3f} double_val={ddv:.3f} gap={gap:+.3f}")
print("ok")

## Part C. The toy mechanism of Fig.1: single vs double estimator

The mechanism in pure isolation: no RL, no networks. The true values of all actions are equal (say, 0), and the estimates are independent Gaussian noise. Show that the single (max) estimator overestimates and that the overestimation grows with the number of actions, while the double estimator is unbiased.

Two independent sets of estimates: you **select** the action with the first and **evaluate** it with the second. This is the decoupling from the theory section, applied to pure noise.

In [ ]:
def overestimation_mechanism(n_actions=(2, 4, 8), trials=20000, seed=123):
    """Paper Sec.3 / Fig.1 mechanism in isolation. True Q*(a)=0 for every action;
    estimates are independent standard-normal noise. Contrast two estimators of
    max_a Q*(a) (whose true value is 0):
      - single estimator: one noisy table both picks and scores the action;
      - double estimator: one noisy table picks the action, a SECOND independent
        noisy table scores it.
    Report the mean (over `trials`) of each estimator for each action count, i.e. each
    estimator's empirical bias away from the true value 0.

    Args:
        n_actions: iterable[int] — action-count settings to sweep.
        trials:    int — Monte-Carlo trials per setting.
        seed:      int — RNG seed (use numpy default_rng).
    Returns:
        dict {n_act: (single_bias, double_bias)} of floats.
    Note: the two estimate tables must be independent draws.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# Calibrated to the real verify numbers: single(max) bias by #actions(2,4,8) ~ +0.571/+1.029/+1.421,
# double bias ~ 0 (<=0.009). Assert the ROBUST direction with generous slack.
mech = overestimation_mechanism()
biases = [mech[n][0] for n in (2, 4, 8)]
print("single(max) bias by #actions (2,4,8):", [round(b, 3) for b in biases])
print("double bias by #actions:", [round(mech[n][1], 3) for n in (2, 4, 8)])
assert all(b > 0.2 for b in biases), "single(max) estimator must overestimate (bias>0)"
assert biases[0] < biases[1] < biases[2], "overestimation must grow with #actions (Fig.1)"
for n in (2, 4, 8):
    assert abs(mech[n][1]) < 0.05, f"double estimator must be ~unbiased (n={n})"
print("ok")

## Part D. Assembling the ablation: train both arms, measure the effect

Now the real run. We train vanilla and Double via `train(double=...)` on several seeds (the paper's effect is distributional; a single seed is just noise), measure return and overestimation for each arm, and compare against the random floor.

The run takes a few minutes on CPU.

In [ ]:
def run_arm(double, seeds):
    """Train one arm across seeds; return per-seed return and per-seed value-overestimation."""
    rets, overs = [], []
    for sd in seeds:
        online, _, _ = train(double=double, seed=sd)
        m, _ = eval_return(greedy_policy(online), episodes=20)
        _, _, over = value_overestimation(online)
        rets.append(m); overs.append(over)
    return rets, overs

SEEDS = (0, 1, 2)
print(f"device={DEVICE} env={ENV_ID} seeds={SEEDS}")
print("training vanilla DQN (max-operator target) across seeds ...")
rets_dqn, over_dqn = run_arm(False, SEEDS)
print("training Double DQN (decoupled select/evaluate target) across seeds ...")
rets_ddqn, over_ddqn = run_arm(True, SEEDS)

ret_rand = eval_return(random_policy())
mean_dqn, std_dqn = float(np.mean(rets_dqn)), float(np.std(rets_dqn))
mean_ddqn, std_ddqn = float(np.mean(rets_ddqn)), float(np.std(rets_ddqn))
over_dqn_m, over_ddqn_m = float(np.mean(over_dqn)), float(np.mean(over_ddqn))

print(f"\nrandom      return = {ret_rand[0]:.1f} +/- {ret_rand[1]:.1f}")
print(f"vanilla DQN return = {mean_dqn:.1f} +/- {std_dqn:.1f}   per-seed {[round(x,1) for x in rets_dqn]}")
print(f"Double DQN  return = {mean_ddqn:.1f} +/- {std_ddqn:.1f}   per-seed {[round(x,1) for x in rets_ddqn]}")
print(f"\nvalue-overestimation  vanilla = {over_dqn_m:+.2f}   Double = {over_ddqn_m:+.2f}   "
      f"reduction = {over_dqn_m - over_ddqn_m:+.2f}")

In [ ]:
# Key figure: return and overestimation per arm.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(["random", "vanilla", "Double"], [ret_rand[0], mean_dqn, mean_ddqn],
          yerr=[ret_rand[1], std_dqn, std_ddqn], color=["#999", "#c44", "#39c"], capsize=4)
ax[0].set_title("episode return (higher better)"); ax[0].set_ylabel("return")
ax[1].bar(["vanilla", "Double"], [over_dqn_m, over_ddqn_m], color=["#c44", "#39c"])
ax[1].axhline(0, color="k", lw=0.8)
ax[1].set_title("value-overestimation = predicted - actual\n(lower better; paper Fig.3)")
plt.tight_layout(); plt.show()

## Part E. Get a feel for the trick: a controlled contrast

This is the reproduction check. The comparison is calibrated to real numbers (seeds 0,1,2; 30k CPU steps): returns random=18.8±13.9, vanilla DQN=64.1±43.8 (per seed 123.9/20.1/48.2), Double DQN=111.5±12.6 (104.3/129.2/101.0); overestimation vanilla=+4.30, Double=−21.45, reduction=+25.75.

The asserts below check the **robust relative effect of the paper**; trivial by-construction facts are left to the earlier tests:
1. `mean_ddqn >= mean_dqn`: decoupling does not lower the return (holds on all 3 seed sets: (0,1,2), (3,4,5), (10,11,12));
2. `over_ddqn_m < over_dqn_m`: Double strictly reduces value overestimation relative to vanilla (holds on all 3 seed sets).

We deliberately do NOT gate on fragile quantities: the std comparison between arms and the absolute sign of vanilla's overestimation both depend on the seed at such a short CPU budget (reported, not asserted). No threshold was tuned to make the tests pass: the relative direction of the effect is exactly what the paper claims and what reproduces. The random floor stays as a lower sanity bound (`ddqn > 3*random`, `dqn > random`).

❓ **Inline.** Look at the per-seed returns of vanilla (123.9/20.1/48.2) versus Double (104.3/129.2/101.0). Double's mean is higher, and its spread across seeds is about a third of vanilla's. Which part of the paper's claim is easiest to see here, and not in the mean return?

In [ ]:
# baseline floor (lower sanity bound only)
print(f"return  random={ret_rand[0]:.1f}  vanilla={mean_dqn:.1f}  Double={mean_ddqn:.1f}")
assert mean_ddqn > 3 * ret_rand[0], "Double DQN must beat random by a wide margin"
assert mean_dqn > ret_rand[0], "even vanilla DQN must beat random"

# EFFECT (A): decoupling does not hurt return (robust direction, holds on all 3 seed sets).
# std reported but NOT asserted (seed-fragile at this short CPU budget).
print(f"effect  return  Double={mean_ddqn:.1f}+/-{std_ddqn:.1f}  vanilla={mean_dqn:.1f}+/-{std_dqn:.1f}")
assert mean_ddqn >= mean_dqn, \
    "Double DQN mean return must be at least vanilla DQN's (decoupling does not hurt)"

# EFFECT (B): the paper's headline, asserted as the ROBUST RELATIVE CONTRAST.
# Double strictly reduces value-overestimation vs vanilla (holds on all 3 seed sets).
# Absolute sign of vanilla's bias is NOT asserted (seed-dependent, reported only).
print(f"effect  value-overestimation  vanilla={over_dqn_m:+.2f}  Double={over_ddqn_m:+.2f}  "
      f"(reduction={over_dqn_m - over_ddqn_m:+.2f})")
assert over_ddqn_m < over_dqn_m, \
    "Double DQN must reduce value-overestimation below vanilla DQN (van Hasselt 2016)"
print("ok — paper effect reproduced: decoupling preserves return and cuts overestimation")

## Part F. Reflection

**What to take into your own work.** Decoupling "who selects" from "who evaluates" is a technique wider than RL. Any estimate of the form "take the best item by a noisy metric and report that same metric" is biased upward: picking the top-k by the same score you then use to measure quality; early stopping on the same validation signal you tuned on. The cure is the same: evaluate the choice on an independent split or run.

❓ **Inline (recognition).** Describe a concrete non-RL situation from your own practice where "selection and evaluation on the same noisy number" produced an optimistic bias. How would you decouple them following the Double DQN recipe: what independent second source of evaluation would you use?

**What is still unclear, and what to read next.** Double DQN removes the overestimation that comes from `max`, but it does not touch exploration noise, function approximation by the network, or the off-policy shift of replay. On large tasks residual overestimation comes back. What to read next: distributional RL (C51/QR-DQN, which model the distribution of the return instead of a point estimate), then dueling networks and PER. Where do you think the overestimation from `max` still bites hardest?

---

If you are stuck on the holes, open the SPOILERS below. Try on your own first: the contracts in the docstrings and the tests pin down the shape of the answer unambiguously.

## SPOILERS: full solutions

Open them one at a time, after a serious attempt.

<details><summary>Part A — <code>td_target</code></summary>

```python
def td_target(reward, next_state, done, online, target, gamma, double):
    with torch.no_grad():
        if double:
            next_actions = online(next_state).argmax(dim=1, keepdim=True)   # selection: online
            next_q = target(next_state).gather(1, next_actions)             # evaluation: target
        else:
            next_q = target(next_state).max(dim=1, keepdim=True).values     # max selects AND evaluates
        y = reward + gamma * next_q * (1.0 - done)
    return y
```
</details>

<details><summary>Part B — <code>value_overestimation</code></summary>

```python
@torch.no_grad()
def value_overestimation(net, gamma=0.99, episodes=20, seed=3000):
    pi = greedy_policy(net)
    env = gym.make(ENV_ID)
    pred, actual = [], []
    for i in range(episodes):
        s, _ = env.reset(seed=seed + i)
        v_pred = net(torch.as_tensor(s, dtype=torch.float32)).max().item()
        done, G, disc = False, 0.0, 1.0
        while not done:
            s, r, term, trunc, _ = env.step(pi(s))
            G += disc * r
            disc *= gamma
            done = term or trunc
        pred.append(v_pred); actual.append(G)
    env.close()
    pred_m, act_m = float(np.mean(pred)), float(np.mean(actual))
    return pred_m, act_m, pred_m - act_m
```
</details>

<details><summary>Part B′ — <code>overestimation_gap</code></summary>

```python
@torch.no_grad()
def overestimation_gap(online, target, states):
    q_t = target(states)
    dqn_val = q_t.max(dim=1).values.mean().item()
    a_star = online(states).argmax(dim=1, keepdim=True)
    double_val = q_t.gather(1, a_star).squeeze(1).mean().item()
    return float(dqn_val), float(double_val), float(dqn_val - double_val)
```
</details>

<details><summary>Part C — <code>overestimation_mechanism</code></summary>

```python
def overestimation_mechanism(n_actions=(2, 4, 8), trials=20000, seed=123):
    rng = np.random.default_rng(seed)
    out = {}
    for n in n_actions:
        qa = rng.normal(0.0, 1.0, size=(trials, n))     # estimator A
        qb = rng.normal(0.0, 1.0, size=(trials, n))     # estimator B (independent)
        single = qa.max(axis=1)                          # max selects AND evaluates on A
        double = qb[np.arange(trials), qa.argmax(axis=1)]  # select on A, evaluate on B
        out[n] = (float(single.mean()), float(double.mean()))
    return out
```
</details>